# Minimal reproduction: `ClassifierMixin`'s `rdf_type` collides with a node's own class-typing triple

Prepared as supporting evidence for a bug report to
[nfdi-de/dcat-ap-plus](https://github.com/nfdi-de/dcat-ap-plus) -- see this
repo's own `ROADMAP.md` for context (found while validating
Health-DCAT-AP-plus/ResHealth-DCAT-AP against real SHACL, tracked as one of
`KNOWN_MERGED_SHAPES_VIOLATIONS`' `type` entries). Directly relevant to
[dcat-ap-plus#84](https://github.com/nfdi-de/dcat-ap-plus/issues/84), which
proposes relying on exactly the mechanism that triggers this bug.

Just two triples: an `Activity` instance with a `title`, nothing else. No
`rdf_type` set. Runs against dcat-ap-plus's *own, real* generated SHACL
(cloned locally, unmodified) -- not a toy schema. Needs the sibling clone
at `repos/dcat-ap-plus` (see this repo's own README.md).

In [1]:
from pathlib import Path
import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

DCAT_AP_PLUS_SCHEMA = Path(
    "C:/Users/remy.ben-messaoud/Documents/python_projects/Health-DCAT-AP-plus/repos/dcat-ap-plus/src/dcat_ap_plus/schema/dcat_ap_plus.yaml"
)
assert DCAT_AP_PLUS_SCHEMA.exists(), "clone https://github.com/nfdi-de/dcat-ap-plus next to this notebook's own repo tree first"

## 1. Generate dcat-ap-plus's own real SHACL, unmodified

In [2]:
shapes_ttl = ShaclGenerator(str(DCAT_AP_PLUS_SCHEMA)).serialize()
shapes_graph = Graph()
shapes_graph.parse(data=shapes_ttl, format="turtle")
print(f"{len(shapes_graph)} triples in dcat-ap-plus's own generated SHACL")

2798 triples in dcat-ap-plus's own generated SHACL


## 2. The two-triple example -- no `rdf_type` set at all

In [3]:
data_ttl = """
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix dcterms: <http://purl.org/dc/terms/> .

<https://example.org/activity/x> a prov:Activity ;
    dcterms:title "Example activity" .
"""
data_graph = Graph()
data_graph.parse(data=data_ttl, format="turtle")
print(data_graph.serialize(format="turtle"))

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix prov: <http://www.w3.org/ns/prov#> .

<https://example.org/activity/x> a prov:Activity ;
    dcterms:title "Example activity" .




## 3. Validate

In [4]:
conforms, results_graph, results_text = pyshacl.validate(
    data_graph,
    shacl_graph=shapes_graph,
    data_graph_format="turtle",
    inference="none",
    advanced=True,
)
print(f"conforms = {conforms}")
print(results_text)

conforms = False
Validation Report
Conforms: False
Results (4):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class schema1:DefinedTerm ; sh:description Literal("The slot to specify the ontology class that is instantiated by an entity.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:order Literal("13", datatype=xsd:integer) ; sh:path rdf:type ]
	Focus Node: <https://example.org/activity/x>
	Value Node: prov:Activity
	Result Path: rdf:type
	Message: Value does not have class schema1:DefinedTerm
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class schema1:DefinedTerm ; sh:description Literal("The slot to specify the ontology class that is instantiated by an entity.") ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:nodeKind sh:IRI ; sh:order Literal("13",

## What happened

The value node in the violation is `prov:Activity` itself -- the node's *own class assertion*, not a value anyone set via `rdf_type`. `ClassifierMixin` mixes an `rdf_type` slot (`slot_uri: rdf:type`) into `Activity`, so the generated shape for `rdf:type` (`sh:class schema:DefinedTerm`) applies to *every* value of that predicate on the node -- including the one asserting the class itself, since LinkML/rdflib_dumper uses the same predicate for both.